<a href="https://colab.research.google.com/github/Comfy05/Football_analytics_portfolio/blob/main/statistics_football.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install mplsoccer
!pip install statsbombpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.8/63.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00


In [33]:
from pandas.core.arrays import period
import pandas as pd
from statsbombpy import sb
from mplsoccer import Pitch, VerticalPitch
from matplotlib.pyplot import scatter
import matplotlib.pyplot as plt

competitions = sb.competitions()
event = sb.events(match_id=22912)

shots = event['type'] == 'Shot'
event_shots = event[shots][['player','location','team','minute', 'shot_outcome', 'shot_statsbomb_xg']]
event_shots['is_on_target'] = event_shots['shot_outcome'].isin(['Goal', 'Saved'])
shot_on_target = event_shots.groupby('team')['is_on_target'].sum().rename('on_target')
total_shots = event_shots.groupby('team').size().rename('all_shots')

print(shot_on_target)
print(total_shots)

team
Liverpool            3
Tottenham Hotspur    8
Name: on_target, dtype: int64
team
Liverpool            11
Tottenham Hotspur     8
dtype: int64
team
Liverpool            14
Tottenham Hotspur    16
Name: all_shots, dtype: int64


In [71]:
faul = event['type'] == 'Foul Committed'
event_offside = event[faul][['player','location','team','minute']]
fauls = event_offside.groupby('team').size()
fauls

,0
team,
Liverpool,10
Tottenham Hotspur,5


In [103]:
offside = event['type'] == 'Offside'
event_offside = event[offside][['player','location','team','minute']]
offsides = event_offside.groupby('team').size().astype(int)
offsides

,0
team,
Tottenham Hotspur,1


In [85]:
possession = event[['possession','possession_team']]
possession = possession.drop_duplicates('possession')
possession_value = possession.groupby('possession_team').size()
possession_value = possession_value/possession_value.sum() * 100
possession_value.round(2)

,0
possession_team,
Liverpool,49.42
Tottenham Hotspur,50.58


In [97]:
from numpy import nan
passing = event['type'] == 'Pass'
event_pass = event[passing][['player','team','minute', 'pass_outcome']]
event_pass['good_pass'] = event_pass['pass_outcome'].isnull()
good_passes = event_pass.groupby('team')['good_pass'].sum()
total_passes = event_pass.groupby('team').size()

print(good_passes)
print(total_passes)

pass_accuracy = (good_passes/total_passes * 100).round(2)
print(pass_accuracy)

team
Liverpool            202
Tottenham Hotspur    447
Name: good_pass, dtype: int64
team
Liverpool            326
Tottenham Hotspur    564
dtype: int64
team
Liverpool            61.96
Tottenham Hotspur    79.26
dtype: float64


In [120]:
match_summary = pd.concat([total_shots, shot_on_target, possession_value, total_passes, pass_accuracy, fauls, offsides], axis=1).fillna(0).round(0)
match_summary.columns = ['all shots', 'shots on target', 'possession', 'passes', 'pass accuracy', 'fauls commited', 'offsides']
match_summary = match_summary.astype({'fauls commited': 'int', 'offsides':'int', 'possession': 'int', 'pass accuracy':'int'})
match_summary_transposed = match_summary.T
match_summary_transposed

,Liverpool,Tottenham Hotspur
all shots,14,16
shots on target,3,8
possession,49,51
passes,326,564
pass accuracy,62,79
fauls commited,10,5
offsides,0,1


array([False,  True])